# 10 — Avaliação de Classificadores e Desbalanceamento

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/10_avaliacao_classificadores.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_avaliacao_classificadores.pdf)
- [Documentação da área](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Medir e interpretar erros por classe, curvas ROC/PR, acurácia balanceada, decisões por limiar e desbalanceamento.

Matriz de confusão descreve erros; precisão (precision) responde quanto das previsões positivas foi positivo; sensibilidade (recall) responde quanto dos verdadeiros positivos foi encontrado. Em problemas raros, acurácia pode ser enganosa. Reamostragem só pode ser aplicada ao treino e dentro de cada fold de validação; `imbalanced-learn` será opcional.


## 2. Fundamentos matemáticos

$\mathrm{precision}=TP/(TP+FP)$, $\mathrm{recall}=TP/(TP+FN)$, $F_1=2PR/(P+R)$ e $\mathrm{balanced\ accuracy}=\frac12(TPR+TNR)$ em problema binário. Valores com denominador zero exigem uma convenção explicitada.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,average_precision_score,balanced_accuracy_score
X,y=make_classification(n_samples=700,n_features=10,weights=[.94,.06],random_state=42)
a,b,c,d=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
for name,model in [('dummy',DummyClassifier(strategy='most_frequent')),('logit',make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)))]:
    model.fit(a,c);pred=model.predict(b)
    print(name,'acc_bal',round(balanced_accuracy_score(d,pred),3),'matriz:\n',confusion_matrix(d,pred))
    print(classification_report(d,pred,zero_division=0))
    if hasattr(model,'predict_proba'):
        proba=model.predict_proba(b)[:,1]
        print('ROC AUC',round(roc_auc_score(d,proba),3),'AP',round(average_precision_score(d,proba),3))

dummy acc_bal 0.5 matriz:
 [[197   0]
 [ 13   0]]
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       197
           1       0.00      0.00      0.00        13

    accuracy                           0.94       210
   macro avg       0.47      0.50      0.48       210
weighted avg       0.88      0.94      0.91       210

ROC AUC 0.5 AP 0.062
logit acc_bal 0.69 matriz:
 [[196   1]
 [  8   5]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       197
           1       0.83      0.38      0.53        13

    accuracy                           0.96       210
   macro avg       0.90      0.69      0.75       210
weighted avg       0.95      0.96      0.95       210

ROC AUC 0.851 AP 0.604


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
from sklearn.metrics import precision_recall_curve
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(a,c)
scores=model.predict_proba(b)[:,1]
precision,recall,thresholds=precision_recall_curve(d,scores)
print('Precisões/recalls calculados:',len(precision),len(recall),'limiares:',len(thresholds))
print('Prevalência no teste:',round(d.mean(),3),'AP baseline aleatório aprox.: prevalência')

Precisões/recalls calculados: 211 211 limiares: 210
Prevalência no teste: 0.062 AP baseline aleatório aprox.: prevalência


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: 30 TP, 10 FP, 20 FN
TP,FP,FN=30,10,20
P=TP/(TP+FP);R=TP/(TP+FN);print('precision',P,'recall',R,'F1',round(2*P*R/(P+R),3))

precision 0.75 recall 0.6 F1 0.667


## 6. Interpretação, limitações e erros comuns

AUC não resume qualidade de probabilidades; calibração é separada. Um threshold precisa ser escolhido na validação segundo custos do problema. Se usar SMOTE, colocá-lo em `imblearn.pipeline.Pipeline` exclusivamente dentro do treinamento/validação; não aplicá-lo ao teste.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- Conferir qual classe é positiva.
- Acurácia alta pode coexistir com recall da classe rara igual a zero.
- Curva PR e average precision são úteis para eventos raros.
- Reamostragem antes do split ou CV causa contaminação.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 07-metricas-de-classificacao.pdf; atividade-2.pdf.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** limiares e desbalanceamento. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **limiares e desbalanceamento** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Matriz de confusão e trade-off do limiar

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.


In [4]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score
y = np.array([0, 0, 0, 1, 1, 1])
score = np.array([.1, .4, .6, .45, .75, .9])
for limite in (.3, .5, .7):
    p = (score >= limite).astype(int)
    print(limite, 'confusão=', confusion_matrix(y, p).tolist(),
          'precisão=', round(precision_score(y,p,zero_division=0),2),
          'revocação=', round(recall_score(y,p,zero_division=0),2))

0.3 confusão= [[1, 2], [0, 3]] precisão= 0.6 revocação= 1.0
0.5 confusão= [[2, 1], [1, 2]] precisão= 0.67 revocação= 0.67
0.7 confusão= [[3, 0], [1, 2]] precisão= 1.0 revocação= 0.67


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **limiares e desbalanceamento** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
